In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
workspace,default


In [0]:
base_path = "/Volumes/workspace/default/etl_demo" 
 
dbutils.fs.mkdirs(f"{base_path}/raw") 
dbutils.fs.mkdirs(f"{base_path}/bronze") 
dbutils.fs.mkdirs(f"{base_path}/silver") 
dbutils.fs.mkdirs(f"{base_path}/gold") 
 
dbutils.fs.ls(base_path) 

[FileInfo(path='dbfs:/Volumes/workspace/default/etl_demo/bronze/', name='bronze/', size=0, modificationTime=1777314355671),
 FileInfo(path='dbfs:/Volumes/workspace/default/etl_demo/gold/', name='gold/', size=0, modificationTime=1777314355672),
 FileInfo(path='dbfs:/Volumes/workspace/default/etl_demo/raw/', name='raw/', size=0, modificationTime=1777314355672),
 FileInfo(path='dbfs:/Volumes/workspace/default/etl_demo/silver/', name='silver/', size=0, modificationTime=1777314355672)]

In [0]:
dbutils.fs.ls(f"{base_path}/raw") 

[FileInfo(path='dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv', name='cogsley_sales.csv', size=2162356, modificationTime=1777314625000)]

In [0]:
raw_file_path = f"{base_path}/raw/cogsley_sales.csv" 
 
bronze_df = ( 
    spark.read 
    .format("csv") 
    .option("header", "true") 
    .option("inferSchema", "true") 
    .option("multiLine", "true") 
    .option("escape", '"') 
    .load(raw_file_path) 
) 
 
bronze_df.printSchema() 
display(bronze_df.limit(10)) 

root
 |-- RowID: integer (nullable = true)
 |-- OrderID: integer (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- OrderMonthYear: date (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Quote: integer (nullable = true)
 |-- DiscountPct: double (nullable = true)
 |-- Rate: integer (nullable = true)
 |-- SaleAmount: double (nullable = true)
 |-- CustomerName: string (nullable = true)
 |-- CompanyName: string (nullable = true)
 |-- Sector: string (nullable = true)
 |-- Industry: string (nullable = true)
 |-- City: string (nullable = true)
 |-- ZipCode: integer (nullable = true)
 |-- State: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- ProjectCompleteDate: date (nullable = true)
 |-- DaystoComplete: integer (nullable = true)
 |-- ProductKey: string (nullable = true)
 |-- ProductCategory: string (nullable = true)
 |-- ProductSubCategory: string (nullable = true)
 |-- Consultant: string (nullable = true)
 |-- Manager: string (nullable = true)
 

RowID,OrderID,OrderDate,OrderMonthYear,Quantity,Quote,DiscountPct,Rate,SaleAmount,CustomerName,CompanyName,Sector,Industry,City,ZipCode,State,Region,ProjectCompleteDate,DaystoComplete,ProductKey,ProductCategory,ProductSubCategory,Consultant,Manager,HourlyWage,RowCount,WageMargin
1914,13729,2009-01-01,2009-01-01,9,1800,0.08,200,1640.96,Matt Bertelsons,The Priceline Group Inc.,Miscellaneous,Business Services,Bowie,20715,Maryland,East,2009-01-03,2,Development - Big Data,Development,Python,Noah Smith,Allen Young,59,1,0.71
4031,28774,2009-01-01,2009-01-01,32,6400,0.1,200,5707.67,Jessica Thornton,Garmin Ltd.,Capital Goods,Industrial Machinery/Components,McKeesport,15131,Pennsylvania,East,2009-01-02,1,Development - Big Data,Development,Market Research,Daniel Tusk,Allen Young,45,1,0.78
1279,9285,2009-01-02,2009-01-01,3,480,0.06,160,447.11,David O'Rourke,Wynn Resorts Limited,Consumer Services,Hotels/Resorts,Prior Lake,55372,Minnesota,Central,2009-01-04,2,Development - Java,Development,Python,Mason Gibson,Josh Martinez,71,1,0.56
5272,37537,2009-01-02,2009-01-01,4,500,0.0,125,495.47,Alan Brumley,Bed Bath & Beyond Inc.,Consumer Services,Home Furnishings,Napa,94559,California,West,2009-01-02,0,Training - Development,Training,Java,William Bufont,Bob Turner,62,1,0.5
5273,37537,2009-01-02,2009-01-01,43,5375,0.07,125,4953.46,Alan Brumley,Bed Bath & Beyond Inc.,Consumer Services,Home Furnishings,Napa,94559,California,West,2009-01-04,2,Training - Development,Training,Strategy,Liam Franklin,Bob Turner,52,1,0.58
5274,37537,2009-01-02,2009-01-01,32,6400,0.05,200,6024.92,Alan Brumley,Bed Bath & Beyond Inc.,Consumer Services,Home Furnishings,Napa,94559,California,West,2009-01-09,7,Development - Big Data,Development,.Net,Emma Watson,Bob Turner,67,1,0.67
6224,44069,2009-01-02,2009-01-01,16,1760,0.09,110,1587.09,Elizabeth Hansen,Fastenal Company,Consumer Services,RETAIL: Building Materials,Montebello,90640,California,West,2009-01-04,2,Development - Python,Development,Business Model,Sophia Dixon,Bob Turner,71,1,0.35
6225,44069,2009-01-02,2009-01-01,43,4730,0.08,110,4312.18,Elizabeth Hansen,Fastenal Company,Consumer Services,RETAIL: Building Materials,Montebello,90640,California,West,2009-01-02,0,Development - Python,Development,SQL,Mia Moore,Bob Turner,51,1,0.54
1074,7909,2009-01-03,2009-01-01,29,3480,0.03,120,3345.1,Alex Grayson,C.H. Robinson Worldwide Inc.,Transportation,Oil Refining/Marketing,Lake Oswego,97035,Oregon,West,2009-01-04,1,Development - Business Logic,Development,Market Research,Abigail Young,Bob Turner,50,1,0.58
1315,9637,2009-01-03,2009-01-01,12,1800,0.08,150,1641.04,Andy Willingham,DIRECTV,Consumer Services,Telecommunications Equipment,Baton Rouge,70802,Louisiana,South,2009-01-05,2,Consulting - Business Model,Consulting,Java,Madison Hill,Frank Mitchell,58,1,0.61


In [0]:
bronze_df.columns 

['RowID',
 'OrderID',
 'OrderDate',
 'OrderMonthYear',
 'Quantity',
 'Quote',
 'DiscountPct',
 'Rate',
 'SaleAmount',
 'CustomerName',
 'CompanyName',
 'Sector',
 'Industry',
 'City',
 'ZipCode',
 'State',
 'Region',
 'ProjectCompleteDate',
 'DaystoComplete',
 'ProductKey',
 'ProductCategory',
 'ProductSubCategory',
 'Consultant',
 'Manager',
 'HourlyWage',
 'RowCount',
 'WageMargin']

In [0]:
from pyspark.sql.functions import current_timestamp, col 
 
bronze_df = ( 
    bronze_df 
    .withColumn("_ingestion_time", current_timestamp()) 
    .withColumn("_source_file", col("_metadata.file_path")) 
) 
 
bronze_path = f"{base_path}/bronze" 
 
( 
    bronze_df.write 
    .format("delta") 
    .mode("overwrite") 
    .option("overwriteSchema", "true") 
    .save(bronze_path) 
)

In [0]:
dbutils.fs.ls(bronze_path) 

[FileInfo(path='dbfs:/Volumes/workspace/default/etl_demo/bronze/_delta_log/', name='_delta_log/', size=0, modificationTime=1777315095528),
 FileInfo(path='dbfs:/Volumes/workspace/default/etl_demo/bronze/part-00000-81ca4316-b8c0-41a3-961d-12dd6953fccf.c000.snappy.parquet', name='part-00000-81ca4316-b8c0-41a3-961d-12dd6953fccf.c000.snappy.parquet', size=297119, modificationTime=1777315044000)]

In [0]:
%sql
DROP TABLE IF EXISTS etl_bronze; 
 
CREATE TABLE etl_bronze 
USING DELTA 
AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/bronze`;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM etl_bronze 
LIMIT 5; 

RowID,OrderID,OrderDate,OrderMonthYear,Quantity,Quote,DiscountPct,Rate,SaleAmount,CustomerName,CompanyName,Sector,Industry,City,ZipCode,State,Region,ProjectCompleteDate,DaystoComplete,ProductKey,ProductCategory,ProductSubCategory,Consultant,Manager,HourlyWage,RowCount,WageMargin,_ingestion_time,_source_file
1914,13729,2009-01-01,2009-01-01,9,1800,0.08,200,1640.96,Matt Bertelsons,The Priceline Group Inc.,Miscellaneous,Business Services,Bowie,20715,Maryland,East,2009-01-03,2,Development - Big Data,Development,Python,Noah Smith,Allen Young,59,1,0.71,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv
4031,28774,2009-01-01,2009-01-01,32,6400,0.1,200,5707.67,Jessica Thornton,Garmin Ltd.,Capital Goods,Industrial Machinery/Components,McKeesport,15131,Pennsylvania,East,2009-01-02,1,Development - Big Data,Development,Market Research,Daniel Tusk,Allen Young,45,1,0.78,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv
1279,9285,2009-01-02,2009-01-01,3,480,0.06,160,447.11,David O'Rourke,Wynn Resorts Limited,Consumer Services,Hotels/Resorts,Prior Lake,55372,Minnesota,Central,2009-01-04,2,Development - Java,Development,Python,Mason Gibson,Josh Martinez,71,1,0.56,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv
5272,37537,2009-01-02,2009-01-01,4,500,0.0,125,495.47,Alan Brumley,Bed Bath & Beyond Inc.,Consumer Services,Home Furnishings,Napa,94559,California,West,2009-01-02,0,Training - Development,Training,Java,William Bufont,Bob Turner,62,1,0.5,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv
5273,37537,2009-01-02,2009-01-01,43,5375,0.07,125,4953.46,Alan Brumley,Bed Bath & Beyond Inc.,Consumer Services,Home Furnishings,Napa,94559,California,West,2009-01-04,2,Training - Development,Training,Strategy,Liam Franklin,Bob Turner,52,1,0.58,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv


In [0]:
from pyspark.sql.functions import col, to_date, when 
 
bronze_delta_df = spark.read.format("delta").load(bronze_path) 
 
silver_df = bronze_delta_df.dropDuplicates() 
 
# Safe type casting 
if "SaleAmount" in silver_df.columns: 
    silver_df = silver_df.withColumn("SaleAmount", col("SaleAmount").cast("double")) 
 
if "Quantity" in silver_df.columns: 
    silver_df = silver_df.withColumn("Quantity", col("Quantity").cast("int")) 
 
if "DiscountPct" in silver_df.columns: 
    silver_df = silver_df.withColumn("DiscountPct", col("DiscountPct").cast("double")) 
 
if "Rate" in silver_df.columns: 
    silver_df = silver_df.withColumn("Rate", col("Rate").cast("double")) 
 
if "HourlyWage" in silver_df.columns: 
    silver_df = silver_df.withColumn("HourlyWage", col("HourlyWage").cast("double")) 
 
if "OrderDate" in silver_df.columns: 
    silver_df = silver_df.withColumn("OrderDate", to_date(col("OrderDate"))) 
 
if "ProjectCompleteDate" in silver_df.columns: 
    silver_df = silver_df.withColumn("ProjectCompleteDate", to_date(col("ProjectCompleteDate"))) 
 
# Add simple data quality flag 
silver_df = silver_df.withColumn( 
    "data_quality_status", 
    when(col("SaleAmount").isNull(), "Invalid SaleAmount") 
    .otherwise("Valid") 
) 
 
silver_path = f"{base_path}/silver" 
 
( 
    silver_df.write 
    .format("delta") 
    .mode("overwrite") 
    .option("overwriteSchema", "true") 
    .save(silver_path) 
) 

In [0]:
dbutils.fs.ls(silver_path) 

[FileInfo(path='dbfs:/Volumes/workspace/default/etl_demo/silver/_delta_log/', name='_delta_log/', size=0, modificationTime=1777315384761),
 FileInfo(path='dbfs:/Volumes/workspace/default/etl_demo/silver/part-00000-1b1e73b4-ed7a-400d-b62c-034539cde0b2.c000.snappy.parquet', name='part-00000-1b1e73b4-ed7a-400d-b62c-034539cde0b2.c000.snappy.parquet', size=311771, modificationTime=1777315340000)]

In [0]:
%sql
DROP TABLE IF EXISTS etl_silver; 
 
CREATE TABLE etl_silver 
USING DELTA 
AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/silver`;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM etl_silver 
LIMIT 5; 

RowID,OrderID,OrderDate,OrderMonthYear,Quantity,Quote,DiscountPct,Rate,SaleAmount,CustomerName,CompanyName,Sector,Industry,City,ZipCode,State,Region,ProjectCompleteDate,DaystoComplete,ProductKey,ProductCategory,ProductSubCategory,Consultant,Manager,HourlyWage,RowCount,WageMargin,_ingestion_time,_source_file,data_quality_status
1914,13729,2009-01-01,2009-01-01,9,1800,0.08,200.0,1640.96,Matt Bertelsons,The Priceline Group Inc.,Miscellaneous,Business Services,Bowie,20715,Maryland,East,2009-01-03,2,Development - Big Data,Development,Python,Noah Smith,Allen Young,59.0,1,0.71,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv,Valid
1194,8710,2009-01-04,2009-01-01,42,4620,0.07,110.0,4257.89,Tamara O'Brill,Symantec Corporation,Technology,Computer Software: Prepackaged Software,Pharr,78577,Texas,Central,2009-01-06,2,Development - Python,Development,Java,Michael Alister,Josh Martinez,68.0,1,0.38,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv,Valid
6953,49730,2009-01-04,2009-01-01,3,375,0.06,125.0,349.32,Dave Hart,VimpelCom Ltd.,Public Utilities,Telecommunications Equipment,Bryan,77803,Texas,Central,2009-01-05,1,Training - Development,Training,Front End Web,Michael Alister,Josh Martinez,68.0,1,0.46,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv,Valid
3981,28451,2009-01-05,2009-01-01,21,2625,0.01,125.0,2575.4,Pauline Wardle,Mattel Inc.,Consumer Non-Durables,Recreational Products/Toys,Norman,73071,Oklahoma,Central,2009-01-07,2,Training - SQL,Training,Java,Noah Smith,Josh Martinez,59.0,1,0.53,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv,Valid
3983,28451,2009-01-05,2009-01-01,33,4950,0.0,150.0,4905.53,Pauline Wardle,Mattel Inc.,Consumer Non-Durables,Recreational Products/Toys,Norman,73071,Oklahoma,Central,2009-01-07,2,Consulting - Strategy,Consulting,Java,Daniel Tusk,Josh Martinez,45.0,1,0.7,2026-04-27T18:37:22.472Z,dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv,Valid


In [0]:
%sql
SELECT data_quality_status, COUNT(*) AS record_count 
FROM etl_silver 
GROUP BY data_quality_status; 

data_quality_status,record_count
Valid,8399


In [0]:
from pyspark.sql.functions import sum as _sum, count as _count, avg as _avg 
 
silver_delta_df = spark.read.format("delta").load(silver_path) 
 
gold_product_revenue_df = ( 
    silver_delta_df 
    .filter(col("data_quality_status") == "Valid") 
    .groupBy("ProductCategory") 
    .agg( 
        _sum("SaleAmount").alias("TotalRevenue"), 
        _count("*").alias("OrderCount"), 
        _avg("SaleAmount").alias("AverageSaleAmount") 
    ) 
    .orderBy(col("TotalRevenue").desc()) 
) 
 
gold_product_path = f"{base_path}/gold/product_revenue" 
 
( 
    gold_product_revenue_df.write 
    .format("delta") 
    .mode("overwrite") 
    .option("overwriteSchema", "true") 
    .save(gold_product_path) 
) 

In [0]:
%sql
DROP TABLE IF EXISTS gold_product_revenue; 
 
CREATE TABLE gold_product_revenue 
USING DELTA 
AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/gold/product_revenue`; 

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM gold_product_revenue 
ORDER BY TotalRevenue DESC;

ProductCategory,TotalRevenue,OrderCount,AverageSaleAmount
Development,1.6437797030000057E7,4610,3565.682652928429
Consulting,7511609.619999989,2065,3637.583351089583
Training,5315590.829999996,1724,3083.2893445475615


In [0]:
gold_region_revenue_df = ( 
    silver_delta_df 
    .filter(col("data_quality_status") == "Valid") 
    .groupBy("Region") 
    .agg( 
        _sum("SaleAmount").alias("TotalRevenue"), 
        _count("*").alias("OrderCount") 
    ) 
    .orderBy(col("TotalRevenue").desc()) 
) 
 
gold_region_path = f"{base_path}/gold/region_revenue" 
 
( 
    gold_region_revenue_df.write 
    .format("delta") 
    .mode("overwrite") 
    .option("overwriteSchema", "true") 
    .save(gold_region_path) 
) 

In [0]:
%sql
DROP TABLE IF EXISTS gold_region_revenue; 
 
CREATE TABLE gold_region_revenue 
USING DELTA 
AS SELECT * FROM delta.`/Volumes/workspace/default/etl_demo/gold/product_revenue`; 

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * 
FROM gold_region_revenue 
ORDER BY TotalRevenue DESC;

ProductCategory,TotalRevenue,OrderCount,AverageSaleAmount
Development,1.6437797030000057E7,4610,3565.682652928429
Consulting,7511609.619999989,2065,3637.583351089583
Training,5315590.829999996,1724,3083.2893445475615


In [0]:
%sql
SELECT * 
FROM gold_product_revenue 
ORDER BY TotalRevenue DESC;

ProductCategory,TotalRevenue,OrderCount,AverageSaleAmount
Development,1.6437797030000057E7,4610,3565.682652928429
Consulting,7511609.619999989,2065,3637.583351089583
Training,5315590.829999996,1724,3083.2893445475615


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT * 
FROM gold_region_revenue 
ORDER BY TotalRevenue DESC;

ProductCategory,TotalRevenue,OrderCount,AverageSaleAmount
Development,1.6437797030000057E7,4610,3565.682652928429
Consulting,7511609.619999989,2065,3637.583351089583
Training,5315590.829999996,1724,3083.2893445475615


Databricks visualization. Run in Databricks to view.

In [0]:
print("Raw folder:") 
display(dbutils.fs.ls(f"{base_path}/raw")) 
 
print("Bronze folder:") 
display(dbutils.fs.ls(f"{base_path}/bronze")) 
 
print("Silver folder:") 
display(dbutils.fs.ls(f"{base_path}/silver")) 
 
print("Gold folder:") 
display(dbutils.fs.ls(f"{base_path}/gold")) 

Raw folder:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/etl_demo/raw/cogsley_sales.csv,cogsley_sales.csv,2162356,1777314625000


Bronze folder:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/etl_demo/bronze/_delta_log/,_delta_log/,0,1777315895553
dbfs:/Volumes/workspace/default/etl_demo/bronze/part-00000-81ca4316-b8c0-41a3-961d-12dd6953fccf.c000.snappy.parquet,part-00000-81ca4316-b8c0-41a3-961d-12dd6953fccf.c000.snappy.parquet,297119,1777315044000


Silver folder:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/etl_demo/silver/_delta_log/,_delta_log/,0,1777315896252
dbfs:/Volumes/workspace/default/etl_demo/silver/part-00000-1b1e73b4-ed7a-400d-b62c-034539cde0b2.c000.snappy.parquet,part-00000-1b1e73b4-ed7a-400d-b62c-034539cde0b2.c000.snappy.parquet,311771,1777315340000


Gold folder:


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/etl_demo/gold/product_revenue/,product_revenue/,0,1777315896941
dbfs:/Volumes/workspace/default/etl_demo/gold/region_revenue/,region_revenue/,0,1777315896941


In [0]:
%sql
SELECT * FROM gold_product_revenue LIMIT 5; 

ProductCategory,TotalRevenue,OrderCount,AverageSaleAmount
Development,1.6437797030000057E7,4610,3565.682652928429
Consulting,7511609.619999989,2065,3637.583351089583
Training,5315590.829999996,1724,3083.2893445475615


Each Layer has different steps through this process. The bronze focuses on extracting the csv file and showing the raw data. The silver focuses on cleaning and making the data look nice. Lastly, the gold focuses on creating the "business metrics".